Exerciţiul 2. Deschiderea unui restaurant fast-food (simulare Monte Carlo)
Doriţi să deschideţi o nouă locaţie fast-food. Clienţii sosesc conform unui proces Poisson cu intensitatea
$\lambda = 20$ clienţi/oră. Timpul necesar pentru plasarea şi plata unei comenzi este exponenţial cu media de $2$
minute, iar timpul de preparare este exponenţial cu media de α minute.
Ipoteze de modelare: există o singură casă şi o singură staţie de gătit; fiecare funcţionează în ordinea
sosirii comenzilor (FIFO). Un client trece mai întâi pe la casă, apoi comanda sa intră în coada staţiei de
gătit. Timpii de servire sunt independenţi între ei şi de sosiri. La începutul orei, sistemul este gol. Pentru
clienţii sosiţi în intervalul $[0, 60]$ minute, urmărim servirea lor completă, chiar dacă aceasta se termină
după minutul $60$.

a) Construiţi în Python o simulare a unei ore: generaţi momentele sosirilor, timpii de plată şi timpii de
preparare; calculaţi pentru fiecare client momentul la care primeşte mâncarea. Explicaţi cum trataţi
cele două cozi.

b) Estimaţi prin simulări repetate probabilitatea ca fiecare client sosit în ora respectivă să primească
mâncarea în cel mult $15$ minute de la sosire. Determinaţi aproximativ cea mai mare valoare a lui $\alpha$
pentru care probabilitatea este de cel puţin 95%.

c) Pentru valoarea lui $\alpha$ obţinută, estimaţi timpul mediu dintre sosirea unui client şi primirea mâncării.
Raportaţi separat, dacă doriţi, timpul mediu petrecut efectiv în aşteptare (fără servire).

d) Precizaţi numărul de simulări, sămânţa aleatoare şi metoda de căutare a lui $\alpha$. Discutaţi pe scurt de ce
estimările se pot modifica de la o rulare la alta.

Indicaţie. Pentru simulare sunt suficiente numpy şi, opţional, scipy.stats. Atenţie: condiţia de 95% se referă
la toţi clienţii dintr-o oră, nu la un client ales aleatoriu.

In [ ]:
from scipy.stats import poisson, expon
import numpy as np

np.random.seed(seed=67)

def run(alpha):
    client_cnt=max(poisson.rvs(mu=20),0)
    arrival_times = sorted(np.linspace(0,60,client_cnt))
    serve_time = 0
    order_times = expon.rvs(loc=2, size=client_cnt)
    preparation_times = expon.rvs(loc=alpha, size=client_cnt)
    # print(arrival_times)
    # print(order_times)
    # print(preparation_times)
    exp_wait_over_15 = 0
    avg_wait_time = 0
    for a, o, p in zip(arrival_times, order_times, preparation_times):
        serve_time = max(a+o,serve_time)+p
        wait_time = max(0, serve_time-a)
        if wait_time > 15:
            exp_wait_over_15 += 1
        avg_wait_time += wait_time
    return client_cnt, avg_wait_time/client_cnt, exp_wait_over_15/client_cnt

# print(run(3))

def b(run_cnt):
    l = 0
    r = 100
    for _ in range(100):
        alpha = (l+r)/2
        good_runs = 0
        for run_id in range(run_cnt):
            client_cnt, avg_wait_time, exp_wait_over_15 = run(alpha)
            if exp_wait_over_15 == 0:
                good_runs += 1

        if good_runs/run_cnt >= 0.95:
            l = alpha
        else:
            r = alpha
    return l

max_alpha = b(1000)
print(f"{max_alpha=}")

def c(alpha, run_cnt):
    total_wait_time = 0
    total_client_cnt = 0
    for run_id in range(run_cnt):
        client_cnt, avg_wait_time, exp_wait_over_15 = run(alpha)
        total_client_cnt += client_cnt
        total_wait_time += avg_wait_time*client_cnt

    print(f"avg wait time for {alpha=}: {total_wait_time/total_client_cnt} ({run_cnt} runs)")

c(max_alpha, 1000)


max_alpha=1.364659878752759
avg wait time for alpha=1.364659878752759: 6.756665247985813 (1000 runs)


d) Saman